# 00 · 安全审计pickle

静态扫描 pickle 字节码，不反序列化。检查有没有能 import 模块、调函数的危险 opcode。

**源文件**：`src/safe_pickle_audit.py`

逐格 `Shift+Enter`。第 1 格是导入，第 2 格是参数（路径已填好），之后是脚本主体。

In [ ]:
# ===== 导入与模块级定义 =====
#!/usr/bin/env python3
"""Statically inspect a pickle stream without deserializing it.

The script records opcode counts and flags any opcode that could introduce or
invoke Python objects. It never prints pickle arguments, because those include
borrower-level personal information.
"""

from __future__ import annotations

import collections
import json
import os
import pickletools
import sys
import time
from pathlib import Path

DANGEROUS_OPCODES = {
    "GLOBAL",
    "STACK_GLOBAL",
    "REDUCE",
    "BUILD",
    "OBJ",
    "INST",
    "NEWOBJ",
    "NEWOBJ_EX",
    "EXT1",
    "EXT2",
    "EXT4",
    "PERSID",
    "BINPERSID",
}

In [ ]:
# ===== 参数：自动定位项目根目录（换台电脑也能跑）=====
import sys
from pathlib import Path

def find_project_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / '3_分析步骤').is_dir() and (c / '5_最终交付包').is_dir():
            return c
    raise RuntimeError('找不到项目根目录。请确认这个 notebook 是在 3_分析步骤/ 文件夹里打开的。')

ROOT     = find_project_root()
DATA_DIR = ROOT / '1_题目与数据'
WORK_DIR = ROOT / '4_中间产物'
OUT_DIR  = WORK_DIR / '我算出来的结果'
print('项目根目录:', ROOT)

sys.argv = ["safe_pickle_audit.py",
    str(DATA_DIR / "Kiva_Loans.pkl"),
    str(Path(str(WORK_DIR)) / "audit/pickle_opcode_audit.json")]
for i, a in enumerate(sys.argv): print(f'  argv[{i}] = {a}')

In [ ]:
# ===== 第 1 段 =====
input_path = Path(sys.argv[1]).resolve()
output_path = Path(sys.argv[2]).resolve()
size = input_path.stat().st_size
counts: collections.Counter[str] = collections.Counter()
dangerous: list[dict[str, int | str]] = []
stop_position: int | None = None
max_text_length = 0
max_bytes_length = 0
next_progress = 100_000_000
started = time.time()

In [ ]:
# ===== 第 2 段 =====
with input_path.open("rb") as stream:
    for op, arg, position in pickletools.genops(stream):
        counts[op.name] += 1
        if op.name in DANGEROUS_OPCODES and len(dangerous) < 100:
            dangerous.append({"opcode": op.name, "position": position})
        if isinstance(arg, str):
            max_text_length = max(max_text_length, len(arg))
        elif isinstance(arg, bytes):
            max_bytes_length = max(max_bytes_length, len(arg))
        if position >= next_progress:
            elapsed = time.time() - started
            print(
                f"scanned={position:,}/{size:,} ({position / size:.1%}) "
                f"elapsed={elapsed:.1f}s",
                flush=True,
            )
            next_progress += 100_000_000
        if op.name == "STOP":
            stop_position = position

In [ ]:
# ===== 第 3 段 =====
result = {
    "input_file": input_path.name,
    "input_size_bytes": size,
    "scan_elapsed_seconds": round(time.time() - started, 3),
    "stop_position": stop_position,
    "trailing_bytes_after_stop": (
        None if stop_position is None else size - stop_position - 1
    ),
    "dangerous_opcode_count": sum(counts[name] for name in DANGEROUS_OPCODES),
    "dangerous_opcodes_first_100": dangerous,
    "max_text_length": max_text_length,
    "max_bytes_length": max_bytes_length,
    "opcode_counts": dict(sorted(counts.items())),
    "safe_for_restricted_primitive_deserialization": (
        stop_position is not None
        and size - stop_position - 1 == 0
        and not dangerous
    ),
    "privacy_note": "Arguments were not logged because the pickle contains borrower-level records.",
}
output_path.parent.mkdir(parents=True, exist_ok=True)
temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
temporary_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
os.replace(temporary_path, output_path)
print(json.dumps(result, indent=2))
#     return 0